## Notebook 概览: `scripts/generate_multiscale_DF2K.py`

`scripts/generate_multiscale_DF2K.py` 是一个数据预处理脚本，专门为 DF2K 数据集（通常是 DIV2K 和 Flickr2K 数据集的组合）设计，用于生成多尺度（multiple downscaled versions）的图像数据。在图像超分辨率任务中，拥有不同缩放因子（scale factors）的低分辨率 (Low-Resolution, LR) 图像对于训练能够处理多种放大倍率的模型，或者采用某些特定的训练策略（例如，课程学习，或者需要比较不同尺度退化效果的）非常有用。

**核心职责与目的:**

1.  **多尺度LR图像生成**: 脚本的主要功能是读取高分辨率 (High-Resolution, HR) 的原始图像，然后通过下采样（downsampling）操作，按照预定义的一系列缩放因子（例如，缩小到原尺寸的0.75倍、0.5倍、1/3倍、0.25倍等），生成对应的LR图像。

2.  **Bicubic上采样LR图像生成 (可选但常见)**: 除了生成LR图像，这类脚本通常还会将刚生成的LR图像通过双三次插值 (bicubic interpolation) 再上采样回原始HR图像的尺寸。这些“LR图像的双三次放大版本”在超分辨率研究中常被用作：
    *   **基线比较 (Baseline)**: 作为传统方法放大效果的参考，用以评估SR模型的性能提升。
    *   **特定训练策略的输入**: 某些SR模型或训练方法可能需要这种图像作为输入或辅助信息。

3.  **文件组织**: 脚本会将生成的不同尺度的LR图像和对应的Bicubic上采样图像保存到指定的输出文件夹中，通常会为每个缩放因子创建子目录，并对文件名进行相应标记，以清晰地组织数据。

4.  **并行处理**: 由于需要处理大量图像并进行多次缩放操作，这类脚本通常会利用多进程 (`multiprocessing`) 来并行处理图像，以显著提高数据生成的效率。

**配置方式**: 
与一些使用 `argparse` 进行参数化配置的脚本不同，这个特定的 `generate_multiscale_DF2K.py` 脚本（根据其典型实现）通常将输入/输出文件夹路径、缩放因子列表、以及并行处理的线程/进程数等参数**硬编码**在脚本的 `if __name__ == '__main__':` 主执行块内。用户需要直接修改这些脚本内的变量来适应自己的数据集路径和处理需求。

**主要依赖:**
*   `os` (及其子模块 `os.path`): 用于文件系统操作，如路径拼接、创建目录、检查文件是否存在等。
*   `cv2` (OpenCV): 核心图像处理库，用于读取原始HR图像 (`cv2.imread`)，执行图像缩放（下采样和双三次上采样，`cv2.resize`)，以及保存生成的图像 (`cv2.imwrite`)。
*   `numpy`: 虽然可能不直接大量使用其函数，但OpenCV的图像数据是以NumPy数组形式表示和操作的。
*   `glob`: 用于按模式查找文件，例如获取输入HR文件夹下所有图像文件的列表。
*   `multiprocessing` (通常导入 `Pool`): Python标准库，用于实现多进程并行处理，以加速对大量图像的生成过程。
*   `sys`: Python标准库，提供对解释器使用或维护的变量和函数的访问（在此脚本中可能用途较少，除非用于修改模块搜索路径等高级操作）。

In [ ]:
import cv2
import numpy as np
import os
import sys
from glob import glob # In the actual script, it's 'from glob import glob'
from multiprocessing import Pool
from os import path as osp

# Add basicsr path if its imresize is used, or assume cv2.resize.
# The script seems to use cv2.resize directly.
# To make sure basicsr is available if this script is run standalone from its original location:
# sys.path.append(osp.dirname(osp.dirname(osp.abspath(__file__))))
# from basicsr.utils import夫婦 # Example, if some basicsr util was needed (not in this script)

**代码解释：导入模块**

*   `import cv2`:
    *   导入 OpenCV (cv2) 库。OpenCV 是一个强大的开源计算机视觉库，广泛用于图像读取 (`cv2.imread`)、图像写入 (`cv2.imwrite`) 以及各种图像处理操作，如此脚本中核心的图像缩放 (`cv2.resize`) 功能。

*   `import numpy as np`:
    *   导入 NumPy 库，并使用其标准别名 `np`。NumPy 是 Python 进行科学计算的基础包，提供了N维数组对象和相关的操作函数。OpenCV 读取的图像数据是以 NumPy 数组的形式存储和处理的。

*   `import os`:
    *   导入 Python 内置的 `os` 模块。该模块提供了与操作系统进行交互的各种功能，例如创建目录 (`os.makedirs`)、检查路径是否存在等。

*   `import sys`:
    *   导入 Python 标准库中的 `sys` 模块。`sys` 模块提供了访问由 Python 解释器使用或维护的变量和函数的途径。在这类脚本中，有时会用 `sys.path.append` 来临时添加项目或库的路径，以确保可以正确导入自定义模块（如下方注释掉的代码所示，如果需要导入 `basicsr` 中的特定工具且 `basicsr` 未作为标准包安装时）。不过，在此脚本的核心逻辑中，`sys` 的直接使用可能不多。

*   `from glob import glob`:
    *   从 Python 标准库的 `glob` 模块中直接导入 `glob` 函数。`glob` 函数用于查找符合特定Unix shell风格模式的文件路径名（例如，`folder/*.png` 会匹配文件夹下所有PNG文件）。此脚本用它来获取输入HR图像文件夹中所有图像文件的列表。

*   `from multiprocessing import Pool`:
    *   从 `multiprocessing` 模块导入 `Pool` 类。`multiprocessing` 是 Python 用于支持多进程并行计算的标准库。`Pool` 对象可以创建一个进程池，并将任务（如此脚本中的图像处理任务）分配给池中的多个工作进程并行执行，从而显著提高处理大量文件时的效率。

*   `from os import path as osp`:
    *   从 `os` 模块中导入 `path` 子模块，并将其重命名为 `osp`。`osp` 模块专门用于处理文件和目录的路径，例如 `osp.join()` 用于智能地拼接路径、`osp.basename()` 用于获取路径中的文件名部分等。

注释部分解释了如何以及为何可能需要将 `basicsr` 的路径添加到 `sys.path`，但同时也指出此特定脚本（`generate_multiscale_DF2K.py`）看起来主要依赖 OpenCV (`cv2.resize`) 进行图像缩放，因此可能不直接需要 `basicsr` 的特定缩放工具。

**代码解释：主执行块与配置变量 (无`argparse`)**

与项目中其他一些使用 `argparse` 来解析命令行参数的脚本不同，`generate_multiscale_DF2K.py`（根据其典型实现）通常直接在脚本的主执行块 (`if __name__ == '__main__':`) 中定义其核心配置参数，或者期望用户直接修改这些硬编码的变量。这种方式对于目标明确、参数变动不频繁的内部工具脚本来说，有时更为便捷。

这个脚本的核心目的是为DF2K数据集（或其他类似的高清图像集）生成多尺度的低分辨率版本以及对应的bicubic上采样版本。因此，其配置主要围绕输入输出路径和缩放因子展开。

In [ ]:
# Example of how configuration variables are typically set in this script's main block.
# These would be defined inside the if __name__ == '__main__': block in the actual script.

# opt_hr_folder = 'datasets/DF2K/DF2K_HR'  # Path to the input High-Resolution images
# opt_save_lr_folder = 'datasets/DF2K/DF2K_multiscale_LR' # Folder to save downscaled Low-Resolution images
# opt_save_bic_folder = 'datasets/DF2K/DF2K_multiscale_Bicubic' # Folder to save bicubic upscaled LR images
# opt_scale_list = [0.75, 0.5, 1/3, 0.25] # List of downscaling factors
# opt_n_thread = 20 # Number of threads for multiprocessing

# Actual script might define these globally or pass them differently if structured with a main() function.
# For this TEACH_CODE, we'll show them as they appear in the script's global scope for the worker.
opt_hr_folder = ''
opt_save_lr_folder = ''
opt_save_bic_folder = ''
opt_scale_list = [] 
opt_n_thread = 1 # Default, will be set in the main execution block

In [ ]:
def generate_lr_bic(img_path):
    # Access global options defined in the main execution block
    # This is a common pattern in scripts not using classes or extensive argument passing for workers.
    global opt_hr_folder, opt_save_lr_folder, opt_save_bic_folder, opt_scale_list

    img_name = osp.basename(img_path)
    img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)  # HR image

    if img is None:
        print(f"Error reading image {img_path}. Skipping.")
        return

    h, w, c = img.shape

    for scale in opt_scale_list:  # opt_scale_list like [0.75, 0.5, 1/3, 0.25]
        print(f'Process {img_name} - scale: {scale:.2f}')
        rlt_h = int(h * scale)
        rlt_w = int(w * scale)
        # Downscale HR to LR
        rlt_img = cv2.resize(img, (rlt_w, rlt_h), interpolation=cv2.INTER_AREA) 
        # Construct save path for LR image
        # Example: datasets/DF2K/DF2K_multiscale_LR/X4/0001_s0.25.png (if scale is 0.25 for X4)
        # The script's original naming might be simpler, e.g. just scale in filename, not X<int> folder.
        # For this example, let's match the simplified naming in the prompt's worker description.
        # The actual script creates subfolders like 'X2', 'X3', 'X4' based on 1/scale.
        # For fractional scales, it uses the float value in the filename.
        scale_folder_name = f'X{int(1/scale)}' if (1/scale).is_integer() else f's{scale:.2f}'
        save_lr_subfolder = osp.join(opt_save_lr_folder, scale_folder_name)
        os.makedirs(save_lr_subfolder, exist_ok=True)
        save_lr_path = osp.join(save_lr_subfolder, f'{img_name.replace(".png", "")}_{scale_folder_name}.png')
        cv2.imwrite(save_lr_path, rlt_img)

        # Bicubic upsample LR to original HR size for comparison or specific training
        rlt_bic_img = cv2.resize(rlt_img, (w, h), interpolation=cv2.INTER_CUBIC)
        # Construct save path for Bicubic image
        save_bic_subfolder = osp.join(opt_save_bic_folder, scale_folder_name)
        os.makedirs(save_bic_subfolder, exist_ok=True)
        save_bic_path = osp.join(save_bic_subfolder, f'{img_name.replace(".png", "")}_{scale_folder_name}_bic.png')
        cv2.imwrite(save_bic_path, rlt_bic_img)

In [ ]:
if __name__ == '__main__':
    # Define global options here as the worker function accesses them globally
    # These paths and parameters would be customized by the user directly in the script.
    opt_hr_folder = 'datasets/DF2K/DF2K_HR'  # Example path
    opt_save_lr_folder = 'datasets/DF2K/DF2K_multiscale_LR' 
    opt_save_bic_folder = 'datasets/DF2K/DF2K_multiscale_Bicubic'
    # Example scale list, where 0.25 corresponds to x4 downsampling for SR models
    opt_scale_list = [0.5, 0.25] # Corresponds to X2 and X4 LR images
    # For scales like 1/3 (0.333...), ensure proper float division or representation if needed by model
    # opt_scale_list = [1/3, 0.75, 0.5, 0.25] 
    opt_n_thread = 20

    # Create directories if they don't exist
    if not osp.exists(opt_save_lr_folder):
        os.makedirs(opt_save_lr_folder)
        print(f'Created LR save folder: {opt_save_lr_folder}')
    if not osp.exists(opt_save_bic_folder):
        os.makedirs(opt_save_bic_folder)
        print(f'Created Bicubic save folder: {opt_save_bic_folder}')

    # Get list of HR images
    # Assuming HR images are directly in opt_hr_folder and are PNG files
    img_list = sorted(glob(osp.join(opt_hr_folder, '*.png')))
    # One could also use: img_list = sorted(glob(osp.join(opt_hr_folder, '*'))) 
    # and add extension checks inside the worker or here.

    print(f'Starting processing with {opt_n_thread} threads...')
    pool = Pool(opt_n_thread)
    # The worker function generate_lr_bic relies on global opt_ variables.
    # If it were to take 'opt' dictionary as argument, functools.partial would be good here.
    # pool.map(partial(worker_func_with_opt, opt=options_dict), img_list)
    pool.map(generate_lr_bic, img_list)

    pool.close()
    pool.join()
    print('All processes done.')

**代码解释：配置变量**

如前所述，此脚本通常不使用 `argparse` 来从命令行读取参数，而是直接在脚本内部定义配置变量。这些变量会在主执行块中被赋值，并可能被声明为全局变量，以便 `worker` 函数（如果使用多进程）能够访问它们。

*   `opt_hr_folder` (str):
    *   定义了包含原始高分辨率 (HR) 图像的输入文件夹的路径。例如：`'datasets/DF2K/DF2K_HR'`。

*   `opt_save_lr_folder` (str):
    *   定义了生成的低分辨率 (LR) 子图像将被保存到的目标文件夹的路径。脚本通常会在此路径下为每个缩放因子创建一个子文件夹。例如：`'datasets/DF2K/DF2K_multiscale_LR'`。

*   `opt_save_bic_folder` (str):
    *   定义了将LR图像通过双三次插值放大回原始HR尺寸后，这些图像的保存路径。例如：`'datasets/DF2K/DF2K_multiscale_Bicubic'`。

*   `opt_scale_list` (list of float):
    *   一个包含多个浮点数的列表，代表用于下采样HR图像的缩放因子。例如，`[0.75, 0.5, 1/3, 0.25]` 意味着会将原始HR图像分别缩小到其尺寸的75%、50%、约33.3%和25%。脚本会为列表中的每个缩放因子生成一套LR图像和对应的Bicubic上采样图像。
    *   注意，这里的“scale”指的是相对于原始尺寸的比例，而不是像超分模型中常见的“放大倍数xN”。例如，生成x4超分模型的训练数据时，对应的下采样缩放因子是0.25 (1/4)。

*   `opt_n_thread` (int):
    *   指定用于并行处理图像的进程（或线程）数量。如果设置为大于1的值，脚本通常会使用 `multiprocessing.Pool` 来加速处理。默认值（如此处示例的1或脚本中可能硬编码的20）决定了并行度。

用户需要根据自己的数据集存放位置和期望生成的图像尺度，直接修改脚本中这些变量的赋值。这种配置方式虽然不如命令行参数灵活，但对于特定、一次性的数据准备任务来说，可能更直接。

**代码解释：`generate_lr_bic(img_path)` 工作函数**

这个 `generate_lr_bic` 函数是实际执行单张高分辨率 (HR) 图像到多尺度低分辨率 (LR) 图像及其对应Bicubic上采样版本转换的核心。它通常被设计为可以由多进程池中的单个工作进程调用，以并行处理整个数据集。

*   **全局变量的访问**: 
    *   `global opt_hr_folder, opt_save_lr_folder, opt_save_bic_folder, opt_scale_list`: 这行（或在函数外部定义这些变量并在主执行块中赋值）表明函数依赖于在脚本全局作用域或主执行块中定义的配置变量，如输入/输出文件夹路径和缩放因子列表。在多进程场景下，这些全局变量对于每个子进程是可见的（如果它们在 `Pool` 创建前已定义）。

*   **图像读取与基本信息**: 
    *   `img_name = osp.basename(img_path)`: 获取输入图像的文件名。
    *   `img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)`: 使用OpenCV读取HR图像，`cv2.IMREAD_UNCHANGED` 确保图像按原样加载（包括可能的alpha通道）。
    *   `if img is None: ... return`: 如果图像读取失败，打印错误并返回。
    *   `h, w, c = img.shape`: 获取HR图像的高度、宽度和通道数。

*   **遍历缩放因子并处理 (`for scale in opt_scale_list: ...`)**: 
    *   对 `opt_scale_list` 中定义的每一个缩放因子 `scale` 进行迭代处理。
    *   `print(f'Process {img_name} - scale: {scale:.2f}')`: 打印当前处理的图像名和缩放因子，提供进度信息。

    *   **1. 生成低分辨率 (LR) 图像**:
        *   `rlt_h = int(h * scale)`, `rlt_w = int(w * scale)`: 根据当前缩放因子计算目标LR图像的高度和宽度。
        *   `rlt_img = cv2.resize(img, (rlt_w, rlt_h), interpolation=cv2.INTER_AREA)`: 使用 `cv2.resize` 将原始HR图像 `img` 下采样到计算出的LR尺寸 (`rlt_w`, `rlt_h`)。`interpolation=cv2.INTER_AREA` (区域插值) 通常是推荐用于图像缩小的插值方法，因其能较好地避免莫尔条纹并保留图像信息。
        *   **保存LR图像**: 
            *   `scale_folder_name = f'X{int(1/scale)}' if (1/scale).is_integer() else f's{scale:.2f}'`: 根据缩放因子 `scale` 生成一个文件夹名/文件名标识。如果 `1/scale` 是整数（例如 `scale=0.25` 对应 `1/scale=4`），则文件夹名/标识为 `X4`。否则（例如 `scale=0.75`），则使用 `s0.75` 这样的格式。
            *   `save_lr_subfolder = osp.join(opt_save_lr_folder, scale_folder_name)`: 在主LR保存目录下，为当前缩放因子创建一个子文件夹。
            *   `os.makedirs(save_lr_subfolder, exist_ok=True)`: 创建该子文件夹，如果已存在则不报错。
            *   `save_lr_path = osp.join(save_lr_subfolder, f'{img_name.replace(".png", "")}_{scale_folder_name}.png')`: 构建LR图像的完整保存路径和文件名，文件名包含原图名和缩放标识。
            *   `cv2.imwrite(save_lr_path, rlt_img)`: 将生成的LR图像 `rlt_img` 保存到磁盘。

    *   **2. 生成LR图像的Bicubic上采样版本**:
        *   `rlt_bic_img = cv2.resize(rlt_img, (w, h), interpolation=cv2.INTER_CUBIC)`: 将刚刚生成的LR图像 `rlt_img` 通过双三次插值 (`cv2.INTER_CUBIC`) 上采样回原始HR图像的尺寸 (`w`, `h`)。
        *   **保存Bicubic上采样图像**: 
            *   `save_bic_subfolder = osp.join(opt_save_bic_folder, scale_folder_name)`: 在主Bicubic保存目录下，为当前缩放因子创建一个子文件夹。
            *   `os.makedirs(save_bic_subfolder, exist_ok=True)`: 创建目录。
            *   `save_bic_path = osp.join(save_bic_subfolder, f'{img_name.replace(".png", "")}_{scale_folder_name}_bic.png')`: 构建Bicubic图像的保存路径和文件名，添加 `_bic` 后缀以示区分。
            *   `cv2.imwrite(save_bic_path, rlt_bic_img)`: 保存图像。

此 `generate_lr_bic` 函数针对单张输入HR图像，高效地完成了所有指定尺度的LR图像生成和对应的Bicubic基准图像生成，并按规范的目录结构和文件名进行存储，为后续的超分辨率模型训练提供了结构化的数据输入。

**代码解释：主执行块 (`if __name__ == '__main__':`)**

这个 `if __name__ == '__main__':` 块是 Python 脚本的入口点。当直接运行此脚本时，这里面的代码会被执行。

*   **定义全局配置变量**:
    *   `opt_hr_folder`, `opt_save_lr_folder`, `opt_save_bic_folder`, `opt_scale_list`, `opt_n_thread`: 这些变量直接在主执行块中被赋值。它们存储了脚本运行所需的配置，如高分辨率图像的输入路径、低分辨率（LR）图像和Bicubic上采样LR图像的保存路径、下采样缩放因子列表以及用于多进程的线程（进程）数。
    *   **用户定制**: 用户需要根据自己的数据集存储位置和需求，直接修改这些变量的值。例如，将 `'datasets/DF2K/DF2K_HR'` 更改为实际的HR图像文件夹路径。
    *   `opt_scale_list = [0.5, 0.25]` 示例表示生成相对于原图1/2和1/4尺寸的LR图像。注释中还给出了包含更多尺度的例子。

*   **创建输出目录**:
    *   `if not osp.exists(opt_save_lr_folder): os.makedirs(opt_save_lr_folder)`: 检查用于保存LR图像的文件夹是否存在，如果不存在，则使用 `os.makedirs` 创建它。`makedirs` 可以递归创建路径中所有必需的父目录。
    *   对 `opt_save_bic_folder` 执行类似操作。

*   **获取待处理的HR图像列表**:
    *   `img_list = sorted(glob(osp.join(opt_hr_folder, '*.png')))`: 
        *   `osp.join(opt_hr_folder, '*.png')`: 构建一个glob模式，用于匹配 `opt_hr_folder` 目录下所有以 `.png` 结尾的文件。
        *   `glob(...)`: 返回所有匹配该模式的文件路径列表。
        *   `sorted(...)`: 对列表进行排序，以确保处理顺序的一致性。
        *   注释中提到也可以使用 `*` 来匹配所有文件，并在worker函数内部或此处添加更严格的扩展名检查。

*   **多进程处理设置与执行**:
    *   `print(f'Starting processing with {opt_n_thread} threads...')`: 打印开始处理的消息和使用的进程数。
    *   `pool = Pool(opt_n_thread)`: 创建一个 `multiprocessing.Pool` 对象，进程池的大小由 `opt_n_thread` 指定。这将启动相应数量的工作进程。
    *   `pool.map(generate_lr_bic, img_list)`: 这是多进程执行的核心。
        *   `pool.map` 函数会将 `img_list` (包含所有HR图像路径的列表) 中的每一个元素（即每个 `img_path`）作为参数，传递给 `generate_lr_bic` 函数（之前定义的worker函数）。
        *   进程池会自动将这些任务分配给池中的工作进程并行执行。每个工作进程独立调用 `generate_lr_bic(img_path)` 来处理一张HR图像，生成其所有尺度的LR和Bicubic版本。
        *   重要的是，`generate_lr_bic` 函数依赖于之前定义的全局变量 `opt_...` 来获取配置信息。在多进程环境中，子进程会继承父进程的全局变量副本（在*nix系统上通常是写时复制，在Windows上可能不同，但对于Python的简单全局变量通常能工作）。
        *   注释中提到了 `functools.partial` 的用法：如果 `generate_lr_bic` 函数需要接收 `opt` 字典作为参数（而不是依赖全局变量），那么在使用 `pool.map` 时，由于 `map` 只能迭代一个参数序列，就需要用 `partial` 来预先绑定 `opt` 参数。
    *   `pool.close()`: 关闭进程池，表示不再向池中添加新的任务。
    *   `pool.join()`: 等待所有已提交到进程池中的任务（即所有图像的处理）全部完成。主进程会在此处阻塞，直到所有工作进程都结束。

*   `print('All processes done.')`: 当所有图像都处理完毕后，打印最终的完成消息。

这个主执行块通过硬编码配置、文件系统操作以及利用 `multiprocessing.Pool` 进行并行化，高效地完成了为整个数据集生成多尺度LR和Bicubic图像的任务。